# V2 - Phase 3 - Baseline iteratif (100K rows)

**Objectif.** Sur 100 000 lignes echantillonnees de `features/company_year_features_v2/`, entrainer deux baselines (LogReg + HGB par defaut) et confirmer empiriquement que les features V2 produisent un signal superieur a V1 leakage-free. C'est un *gate* avant de lancer le sweep complet de la Phase 4.

**Pourquoi.** En V1, les premiers runs ont tourne sur 2M lignes (10 min par fit), ce qui a decourage l'iteration et permis l'accumulation de 7 runs LogReg avant de detecter la fuite INSEE. Phase 3 etablit la discipline: a 100K lignes (<1 min par fit), on teste une intuition en quelques minutes au lieu de quelques heures.

**Critere pour passer a la Phase 4.**

- HGB V2 doit obtenir AP >= 0.18 sur 100K rows (gain >= 2.5 pp sur V1 Run 8 = 0.155).
- LogReg V2 doit afficher un signal non-trivial (AP > 0.06, V1 Run 3 etait a 0.061 leakage-free).

Si HGB V2 < 0.16 sur 100K, audit avant Phase 4: la jointure periode-aware n'apporte pas le gain attendu.


## 1. Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-v2'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'

FEATURES_V2 = f'{DATA_LAKE}/features/company_year_features_v2'
LABELS_V2   = f'{DATA_LAKE}/features/risk_labels_v2'

ARTIFACTS_DRIVE = f'{DRIVE_ROOT}/ml-artifacts/v2/phase_3_baseline_100k'
Path(ARTIFACTS_DRIVE).mkdir(parents=True, exist_ok=True)

# Sampling / split params
HASH_SALT      = 'v2_phase3_baseline'
TARGET_COL     = 'continuity_risk_12m_label'   # composite target -- direct comparison with V1
SAMPLE_SIZE    = 100_000
TRAIN_YEARS    = (2017, 2022)                  # inclusive
TEST_YEARS     = (2023, 2023)                  # 2024 has incomplete labels (need 12m forward)

print('FEATURES_V2     =', FEATURES_V2)
print('LABELS_V2       =', LABELS_V2)
print('ARTIFACTS_DRIVE =', ARTIFACTS_DRIVE)


In [ ]:
import os, subprocess

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

!pip install -q -r collabs/requirements-colab.txt


## 2. Charger features + labels et echantillonner 100K rows

L'echantillonnage est *hash-deterministe* sur `siren` pour que LogReg et HGB voient exactement le meme set, et pour que les futures relances (debug, ablation) tirent le meme echantillon.


In [ ]:
import duckdb, hashlib, numpy as np, pandas as pd

con = duckdb.connect()
con.execute(f"PRAGMA threads={os.cpu_count() or 4}")
con.execute("PRAGMA memory_limit='25GB'")

# Total population in V2 features
total = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{FEATURES_V2}/**/*.parquet')"
).fetchone()[0]
print(f'Total rows in V2 features : {total:,}')

# Hash-deterministic threshold for the 100K sample
threshold = max(1, total // SAMPLE_SIZE)
print(f'Hash modulo threshold      : 1 in {threshold:,} (target {SAMPLE_SIZE:,} rows)')


In [ ]:
# Pull features + labels for sample, train years and test year only.
# The hash filter uses a fixed salt so re-runs draw the same subset.
sample_sql = f"""
WITH features AS (
    SELECT *
    FROM read_parquet('{FEATURES_V2}/**/*.parquet')
    WHERE prediction_year BETWEEN {TRAIN_YEARS[0]} AND {TEST_YEARS[1]}
),
labels AS (
    SELECT siren, prediction_year, {TARGET_COL}
    FROM read_parquet('{LABELS_V2}/**/*.parquet')
    WHERE prediction_year BETWEEN {TRAIN_YEARS[0]} AND {TEST_YEARS[1]}
      AND {TARGET_COL} IS NOT NULL
),
joined AS (
    SELECT f.*, l.{TARGET_COL} AS target
    FROM features f
    INNER JOIN labels l USING (siren, prediction_year)
)
SELECT *
FROM joined
WHERE (
    ABS(hash(siren || '|{HASH_SALT}')) % {threshold}
) = 0
"""
df = con.execute(sample_sql).df()
print(f'Sample size              : {len(df):,}')
print(f'Positive rate            : {df["target"].mean():.4%}')
print(f'Per-year row counts      :')
print(df.groupby('prediction_year').size().to_string())
print(f'Per-year positive rate   :')
print(df.groupby('prediction_year')['target'].mean().to_string())


## 3. Train/test split temporel

- **Train**: 2017-2022.
- **Test**: 2023 (2024 a des etiquettes 12-mois non mures).


In [ ]:
train = df[(df['prediction_year'] >= TRAIN_YEARS[0]) & (df['prediction_year'] <= TRAIN_YEARS[1])].copy()
test  = df[(df['prediction_year'] >= TEST_YEARS[0])  & (df['prediction_year'] <= TEST_YEARS[1])].copy()
print(f'Train rows : {len(train):,}  positives={train["target"].sum():,}  rate={train["target"].mean():.4%}')
print(f'Test  rows : {len(test):,}  positives={test["target"].sum():,}  rate={test["target"].mean():.4%}')


## 4. Preparer les features

- Colonnes a ne PAS utiliser comme features (identifiants, dates, fuites): `siren`, `prediction_year`, `prediction_date`, `creation_date`, `company_name`, `target`.
- Categorielles INSEE: `activity_code`, `legal_category_code`, `employee_size_bracket`, `administrative_status_at_cutoff`. C'est l'apport de V2.
- Numeriques: tout le reste (counts BODACC, INPI, financiers, age).
- Booleen `has_*` -> int.


In [ ]:
ID_COLS = ['siren', 'prediction_year', 'prediction_date', 'creation_date', 'company_name', 'target']
CAT_COLS = ['activity_code', 'legal_category_code', 'employee_size_bracket', 'administrative_status_at_cutoff']

feat_cols = [c for c in df.columns if c not in ID_COLS]
num_cols  = [c for c in feat_cols if c not in CAT_COLS]
print(f'#features = {len(feat_cols)}  ({len(num_cols)} num, {len(CAT_COLS)} cat)')

# Booleans -> int
for c in num_cols:
    if df[c].dtype == bool:
        train[c] = train[c].astype('int8')
        test[c]  = test[c].astype('int8')

X_train = train[feat_cols]
y_train = train['target'].astype('int8')
X_test  = test[feat_cols]
y_test  = test['target'].astype('int8')


## 5. Baseline 1 - Logistic Regression (OneHot + StandardScaler)

Comparaison directe avec V1 Run 3 (AP = 0.061 sur 2M, identite exclue). On attend au minimum AP > 0.06 ; un gain franc indiquerait que les features d'identite V2 apportent un signal lineaire.


In [ ]:
import time, json as _json
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  StandardScaler()),
        ]), num_cols),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='__missing__')),
            ('ohe',    OneHotEncoder(handle_unknown='ignore', min_frequency=20)),
        ]), CAT_COLS),
    ],
    remainder='drop',
)

logreg = Pipeline([
    ('prep', preprocessor),
    ('clf',  LogisticRegression(max_iter=1000, n_jobs=-1, solver='lbfgs')),
])

t0 = time.time()
logreg.fit(X_train, y_train)
t_fit = time.time() - t0

y_score = logreg.predict_proba(X_test)[:, 1]
y_pred  = (y_score >= 0.5).astype(int)

logreg_metrics = {
    'model'       : 'LogReg (OHE + Scaler)',
    'fit_seconds' : round(t_fit, 1),
    'ap'          : float(average_precision_score(y_test, y_score)),
    'auc'         : float(roc_auc_score(y_test, y_score)),
    'f1_at_0_5'   : float(f1_score(y_test, y_pred)),
    'precision_at_0_5': float(precision_score(y_test, y_pred, zero_division=0)),
    'recall_at_0_5'   : float(recall_score(y_test, y_pred, zero_division=0)),
    'test_positive_rate': float(y_test.mean()),
}
print(_json.dumps(logreg_metrics, indent=2))


## 6. Baseline 2 - HistGradientBoosting (defaut, support natif categorielles)

Comparaison directe avec V1 Run 8 (AP = 0.155 sur 2M, HGB default). C'est le **critere de validation** de la Phase 3.


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# HGB native categorical: needs int/categorical dtypes for cat cols
X_train_h = X_train.copy()
X_test_h  = X_test.copy()
cat_indices = []
for c in CAT_COLS:
    X_train_h[c] = X_train_h[c].astype('category')
    X_test_h[c]  = X_test_h[c].astype('category')
    # Align test categories to train
    X_test_h[c]  = X_test_h[c].cat.set_categories(X_train_h[c].cat.categories)
    cat_indices.append(X_train_h.columns.get_loc(c))

hgb = HistGradientBoostingClassifier(
    categorical_features=cat_indices,
    random_state=42,
)

t0 = time.time()
hgb.fit(X_train_h, y_train)
t_fit = time.time() - t0

y_score = hgb.predict_proba(X_test_h)[:, 1]
y_pred  = (y_score >= 0.5).astype(int)

hgb_metrics = {
    'model'       : 'HGB (defaut, native cat)',
    'fit_seconds' : round(t_fit, 1),
    'ap'          : float(average_precision_score(y_test, y_score)),
    'auc'         : float(roc_auc_score(y_test, y_score)),
    'f1_at_0_5'   : float(f1_score(y_test, y_pred)),
    'precision_at_0_5': float(precision_score(y_test, y_pred, zero_division=0)),
    'recall_at_0_5'   : float(recall_score(y_test, y_pred, zero_division=0)),
    'test_positive_rate': float(y_test.mean()),
}
print(_json.dumps(hgb_metrics, indent=2))


## 7. Tableau comparatif V1 vs V2

V1 baselines (a echantillon 2M, identite exclue) viennent des Run 3 / Run 8 documentes dans `docs/ml_experiment_tracking_report.md`. Pour comparer a iso-echantillon, refaire V1 a 100K serait l'ideal mais hors-perimetre de cette phase ; les valeurs V1 ici sont indicatives.


In [ ]:
rows = [
    {'split': 'V1 (2M, identite exclue)', 'model': 'LogReg', 'ap_or_note': '0.061 (Run 3)'},
    {'split': 'V1 (2M, identite exclue)', 'model': 'HGB',    'ap_or_note': '0.155 (Run 8)'},
    {'split': f'V2 (100K, periode-aware)', 'model': 'LogReg', 'ap_or_note': f"{logreg_metrics['ap']:.3f}"},
    {'split': f'V2 (100K, periode-aware)', 'model': 'HGB',    'ap_or_note': f"{hgb_metrics['ap']:.3f}"},
]
comp = pd.DataFrame(rows)
print(comp.to_string(index=False))

verdict_lines = []
hgb_ap = hgb_metrics['ap']
if hgb_ap >= 0.18:
    verdict_lines.append(f'[OK] HGB V2 AP = {hgb_ap:.3f} >= 0.18 (critere passe). Passer a la Phase 4.')
elif hgb_ap >= 0.16:
    verdict_lines.append(f'[BORDERLINE] HGB V2 AP = {hgb_ap:.3f}. Au-dessus de V1 mais sous le critere de 0.18.')
    verdict_lines.append('  Possibles causes: bruit a 100K, ou apport identite plus faible que prevu.')
    verdict_lines.append('  Decision: relancer a 500K avant la Phase 4 pour confirmer le signal.')
else:
    verdict_lines.append(f'[KO] HGB V2 AP = {hgb_ap:.3f} < 0.16. Audit requis avant Phase 4.')
    verdict_lines.append('  Verifier: % NaN par feature, distribution target, qualite jointure identite.')

print()
for line in verdict_lines:
    print(line)


## 8. Sauvegarder les artefacts


In [ ]:
import json as _json

manifest = {
    'phase'           : 'phase_3_baseline_100k',
    'target_col'      : TARGET_COL,
    'hash_salt'       : HASH_SALT,
    'sample_size'     : SAMPLE_SIZE,
    'actual_sample'   : len(df),
    'train_years'     : TRAIN_YEARS,
    'test_years'      : TEST_YEARS,
    'n_features'      : len(feat_cols),
    'n_num_features'  : len(num_cols),
    'n_cat_features'  : len(CAT_COLS),
    'logreg'          : logreg_metrics,
    'hgb'             : hgb_metrics,
    'v1_baselines'    : {'logreg_ap_2m_run3': 0.061, 'hgb_ap_2m_run8': 0.155},
}

out_path = Path(ARTIFACTS_DRIVE) / 'manifest.json'
out_path.write_text(_json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Wrote {out_path}')
print(_json.dumps(manifest, indent=2))

con.close()


## 9. Mettre a jour le journal de phase

Ouvrir `docs/v2/v2_phase_log.md`, section Phase 3, et remplir la table avec:

- LogReg AP V2 (100K) = `logreg_metrics['ap']`
- HGB AP V2 (100K) = `hgb_metrics['ap']`
- Verdict: passer a la Phase 4 ssi HGB AP >= 0.18.

Puis decider:

- Si OK -> Phase 4: entrainer un modele HGB par etiquette (5 modeles) sur le full V2.
- Si borderline -> relancer Phase 3 a 500K rows pour confirmer.
- Si KO -> auditer la qualite des features V2 (% NaN, jointure, distribution target) avant Phase 4.
